# Assault DDQN - HU009C delivery artifacts and technical report

Notebook ejecutable para consolidar la corrida full HU009 en artefactos academicos: modelo compacto, evidencias TensorBoard, evaluacion, video y reporte. El entrenamiento HU009 queda disponible solo como etapa opcional y no se ejecuta por defecto.


## 1. Bootstrap Local -> GitHub -> Colab


In [ ]:
import os

os.environ.setdefault("ASSAULT_BOOTSTRAP_REF", "feature/hu009c-delivery-artifacts")
os.environ.pop("ASSAULT_BOOTSTRAP_COMMIT", None)

os.environ.setdefault("ASSAULT_TRAINING_PROFILE", "full")
os.environ.setdefault("ASSAULT_PROJECT_RUN_ID", "assault_ddqn_" + "full" + "_001")
os.environ.setdefault("ASSAULT_REQUESTED_MODE", "auto")

os.environ.setdefault("ASSAULT_EVALUATION_EPISODES", "10")
os.environ.setdefault("ASSAULT_EVALUATION_EPSILON", "0.0")
os.environ.setdefault("ASSAULT_EXECUTION_MODE", "auto")
# ASSAULT_CHECKPOINT_STEP defaults later to FULL_TRAINING_TARGET_TIMESTEPS
os.environ.setdefault("ASSAULT_COMPACT_MODEL_PATH", "")
os.environ.setdefault("ASSAULT_VIDEO_MAX_STEPS", "1800")


from pathlib import Path
import os


try:
    from google.colab import drive  # type: ignore
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/reinforcement_learning_reto_1")
else:
    BASE = Path.cwd()

os.environ.setdefault("ASSAULT_BOOTSTRAP_REF", "main")
os.environ.setdefault("ASSAULT_MLFLOW_TRACKING_URI", (BASE / "mlruns").as_uri())
os.environ.setdefault("ASSAULT_CHECKPOINT_DIR", str(BASE / "checkpoints"))
os.environ.setdefault("ASSAULT_TENSORBOARD_DIR", str(BASE / "tensorboard"))

print("Persistent storage configured")
print("BASE:", BASE)


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/j-mauro-r/reinforcement_learning_reto_1.git"
COLAB_ROOT = Path("/content/reinforcement_learning_reto_1")
BOOTSTRAP_REF = os.environ.get("ASSAULT_BOOTSTRAP_REF", "main")
BOOTSTRAP_COMMIT = os.environ.get("ASSAULT_BOOTSTRAP_COMMIT") or None
INSTALL_DEPENDENCIES = os.environ.get("ASSAULT_INSTALL_DEPENDENCIES", "1") == "1"


def _running_in_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ImportError:
        return False
    return True


def _git_output(args, cwd):
    return subprocess.check_output(["git", *args], cwd=str(cwd), text=True).strip()


if _running_in_colab():
    if not (COLAB_ROOT / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_ROOT)], check=True)
    subprocess.run(["git", "fetch", "--prune", "origin"], cwd=str(COLAB_ROOT), check=True)
    provisional_ref = BOOTSTRAP_COMMIT or f"origin/{BOOTSTRAP_REF}"
    provisional_sha = _git_output(["rev-parse", "--verify", f"{provisional_ref}^{{commit}}"], COLAB_ROOT)
    subprocess.run(["git", "checkout", "--detach", provisional_sha], cwd=str(COLAB_ROOT), check=True)
    ASSAULT_DIR = COLAB_ROOT / "2_Assault"
else:
    PROJECT_ROOT = Path(_git_output(["rev-parse", "--show-toplevel"], Path.cwd()))
    ASSAULT_DIR = PROJECT_ROOT / "2_Assault"

for path in (ASSAULT_DIR, ASSAULT_DIR.parent):
    value = str(path.resolve())
    if value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

from src.execution_bootstrap import (
    install_project_requirements,
    prepare_execution_environment,
    verify_environment_import,
)

bootstrap = prepare_execution_environment(
    requested_ref=BOOTSTRAP_REF,
    requested_commit=BOOTSTRAP_COMMIT,
    repo_url=REPO_URL,
    colab_root=COLAB_ROOT,
)

PROJECT_ROOT = bootstrap.repo_root
ASSAULT_DIR = bootstrap.assault_dir

if INSTALL_DEPENDENCIES:
    install_project_requirements(bootstrap.requirements_path)

environment_source = verify_environment_import(bootstrap)
bootstrap.as_dict()


In [ ]:
# Optional notebook diagnostics go after the project bootstrap/import cells.
# This placeholder intentionally avoids importing src before sys.path is configured.


## 2. Imports, configuration and tracking


In [ ]:
from pathlib import Path

from src.callbacks import TensorBoardLogger, load_tensorboard_scalars
from src.environment import create_assault_env, get_environment_metadata, validate_frameskip_once
from src.evaluator import evaluate_agent
from src.hu009c_delivery import resolve_hu009c_execution_mode
from src.model_artifact import export_inference_model, load_inference_model
from src.preflight import run_preflight_checks
from src.reporting import plot_training_figures, prepare_training_figures
from src.video import generate_assault_demo_video
from src.session_bootstrap import (
    inspect_experiment_state,
    prepare_training_session,
    update_experiment_state_after_success,
)
from src.tracking import MLflowTracker
from src.training_profiles import (
    assert_training_can_start,
    FULL_TRAINING_TARGET_TIMESTEPS,
    evaluate_full_training_ready,
    resolve_training_profile,
)
from src.training_session import run_training_session
from src.utils import get_runtime_info, load_yaml_config

BASE_CONFIG = load_yaml_config(ASSAULT_DIR / "configs" / "ddqn_config.yaml")
TRAINING_PROFILE = os.environ.get("ASSAULT_TRAINING_PROFILE", "smoke").strip().lower()
_TARGET_OVERRIDE = os.environ.get("ASSAULT_TARGET_TIMESTEPS")
profile_context = resolve_training_profile(
    BASE_CONFIG,
    TRAINING_PROFILE,
    target_timesteps=int(_TARGET_OVERRIDE) if _TARGET_OVERRIDE else None,
)
config = profile_context.config
seed = int(config["reproducibility"]["seed"])
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ASSAULT_DIR:", ASSAULT_DIR)
print("BOOTSTRAP_REF:", BOOTSTRAP_REF)
print("BOOTSTRAP_COMMIT:", BOOTSTRAP_COMMIT or "<none>")
print("EXECUTED_SHA:", bootstrap.resolved_sha)
print("src.environment:", environment_source)
print("TRAINING_PROFILE=", profile_context.name)
print("PROFILE_TARGET_TIMESTEPS=", profile_context.target_timesteps)
print("FULL_TRAINING_TARGET_TIMESTEPS=", FULL_TRAINING_TARGET_TIMESTEPS)
print("REPLAY_BUFFER_CAPACITY=", config["replay_buffer"]["capacity"])
print("REPLAY_BUFFER_ESTIMATED_GIB=", round(profile_context.replay_buffer_memory.total_gib, 3))
config


## 3. Runtime and hardware


In [ ]:
runtime_info = get_runtime_info()
runtime_info


## 4. HU002 environment contract


In [ ]:
train_env = create_assault_env(config, mode="train", seed=seed)
eval_env = create_assault_env(config, mode="eval", seed=seed + 1)

obs, info = train_env.reset(seed=seed)
metadata = get_environment_metadata(train_env, config, mode="train", seed=seed)

print("Observation shape:", obs.shape)
print("Observation dtype:", obs.dtype)
print("Action space:", train_env.action_space)
print("Action meanings:", train_env.unwrapped.get_action_meanings())
print("Initial info:", info)
print("Metadata:", metadata)


## 5. HU002 autovalidations


In [ ]:
assert obs.shape == (4, 84, 84)
assert str(obs.dtype) == "uint8"
assert train_env.action_space.n == 7
assert train_env.observation_space.shape == eval_env.observation_space.shape
assert train_env.observation_space.dtype == eval_env.observation_space.dtype
assert validate_frameskip_once(train_env, expected_frameskip=4, steps=5)

obs, info = train_env.reset(seed=seed)
for step in range(100):
    action = int(train_env.action_space.sample())
    obs, reward, terminated, truncated, info = train_env.step(action)
    assert obs.shape == (4, 84, 84)
    assert str(obs.dtype) == "uint8"
    if terminated or truncated:
        obs, info = train_env.reset()

print("HU002 validations passed.")
train_env.close()
eval_env.close()


## 6. HU004 preflight gate


In [ ]:
preflight_report = run_preflight_checks(config)
print(preflight_report.format_summary())
preflight_report.as_dict()


## 7. Abort if preflight fails


In [ ]:
if not preflight_report.ready_for_training:
    raise RuntimeError("READY_FOR_TRAINING=False; HU005 training aborted.")
print("READY_FOR_TRAINING=True")


## 8. HU009 profile and automated session bootstrap


In [ ]:
def _optional_env(name: str, default=None):
    value = os.environ.get(name)
    if value is None or not value.strip():
        return default
    return value.strip()


mlflow_config = config.get("mlflow", {})
evaluation_config = config.get("evaluation", {})
PROJECT_RUN_ID = _optional_env("ASSAULT_PROJECT_RUN_ID", config["checkpointing"]["run_id"] + f"_{profile_context.name}_hu009")
TARGET_TIMESTEPS = profile_context.target_timesteps
CHECKPOINT_STEP = int(os.environ.get("ASSAULT_CHECKPOINT_STEP") or TARGET_TIMESTEPS)
SOURCE_CHECKPOINT_PATH = Path(
    os.environ.get("ASSAULT_SOURCE_CHECKPOINT_PATH")
    or (Path(os.environ["ASSAULT_CHECKPOINT_DIR"]) / PROJECT_RUN_ID / f"checkpoint_step_{CHECKPOINT_STEP:06d}.pt")
)
REQUESTED_MODE = _optional_env("ASSAULT_REQUESTED_MODE", "auto").lower()
RESUME_MODE = _optional_env("ASSAULT_RESUME_MODE", "resume_full")
EVALUATION_EPISODES = int(_optional_env("ASSAULT_EVALUATION_EPISODES", evaluation_config.get("episodes", 2)))
EVALUATION_EPSILON = float(_optional_env("ASSAULT_EVALUATION_EPSILON", evaluation_config.get("epsilon", 0.0)))
_evaluation_max_steps = _optional_env("ASSAULT_EVALUATION_MAX_STEPS", evaluation_config.get("max_steps_per_episode"))
EVALUATION_MAX_STEPS = int(_evaluation_max_steps) if _evaluation_max_steps is not None else None

legacy_run_training = os.environ.get("ASSAULT_RUN_TRAINING")
EXECUTION_MODE = _optional_env("ASSAULT_EXECUTION_MODE")
if EXECUTION_MODE is None:
    if legacy_run_training == "1":
        EXECUTION_MODE = "train"
    elif legacy_run_training == "0":
        EXECUTION_MODE = "delivery"
    else:
        EXECUTION_MODE = "auto"

selected_runtime = "Google Colab" if _running_in_colab() else "local"
selected_device = "cuda" if __import__("torch").cuda.is_available() else "cpu"

session_kwargs = {
    "base_path": BASE,
    "project_run_id": PROJECT_RUN_ID,
    "target_timesteps": TARGET_TIMESTEPS,
    "requested_mode": REQUESTED_MODE,
    "config": config,
    "checkpoint_root": os.environ.get("ASSAULT_CHECKPOINT_DIR"),
    "tensorboard_root": os.environ.get("ASSAULT_TENSORBOARD_DIR"),
    "tracking_uri": os.environ.get("ASSAULT_MLFLOW_TRACKING_URI"),
    "resume_mode": RESUME_MODE,
    "bootstrap_ref": BOOTSTRAP_REF,
    "bootstrap_commit": bootstrap.resolved_sha,
}
delivery_execution = resolve_hu009c_execution_mode(
    run_training=None,
    execution_mode=EXECUTION_MODE,
    project_run_id=PROJECT_RUN_ID,
    target_timesteps=TARGET_TIMESTEPS,
    final_checkpoint_path=SOURCE_CHECKPOINT_PATH,
    prepare_training_session_fn=prepare_training_session,
    prepare_training_session_kwargs=session_kwargs,
)
session_context = delivery_execution.session_context
RUN_TRAINING = delivery_execution.training_required
HU009C_POST_TRAINING_READY = delivery_execution.hu009c_post_training_ready
TRAINING_SESSION_BOOTSTRAP_SKIPPED = delivery_execution.training_session_bootstrap_skipped
AUTO_RESOLUTION = delivery_execution.auto_resolution
TRAINING_SKIPPED_FINAL_CHECKPOINT_EXISTS = bool(AUTO_RESOLUTION == "DELIVERY" and SOURCE_CHECKPOINT_PATH.exists())

if RUN_TRAINING:
    full_training_gate = evaluate_full_training_ready(
        profile_context=profile_context,
        session_context=session_context,
        preflight_report=preflight_report,
        runtime_info=runtime_info,
        observation_shape=tuple(obs.shape),
        observation_dtype=str(obs.dtype),
        action_space=str(metadata.action_space),
        runtime=selected_runtime,
        device=selected_device,
    )
    FULL_TRAINING_READY = full_training_gate.ready
    assert_training_can_start(profile_context, full_training_gate)

    RUN_ID = session_context.project_run_id
    CHECKPOINT_DIR = session_context.checkpoint_root
    TENSORBOARD_DIR = session_context.tensorboard_root
    MLFLOW_TRACKING_MODE = session_context.tracking_mode
    MLFLOW_RUN_ID = session_context.mlflow_run_id
    MLFLOW_SESSION_ID = session_context.tracking_session_id
    CHECKPOINT_INPUT_REFERENCE = str(session_context.checkpoint_input) if session_context.checkpoint_input else None
    SESSION_TARGET_TIMESTEPS = session_context.target_timesteps
else:
    full_training_gate = None
    FULL_TRAINING_READY = None
    RUN_ID = PROJECT_RUN_ID
    CHECKPOINT_DIR = Path(os.environ["ASSAULT_CHECKPOINT_DIR"])
    TENSORBOARD_DIR = Path(os.environ["ASSAULT_TENSORBOARD_DIR"])
    MLFLOW_TRACKING_MODE = None
    MLFLOW_RUN_ID = None
    MLFLOW_SESSION_ID = None
    CHECKPOINT_INPUT_REFERENCE = None
    SESSION_TARGET_TIMESTEPS = TARGET_TIMESTEPS

print("ASSAULT_EXECUTION_MODE=", EXECUTION_MODE)
print("AUTO_RESOLUTION=", AUTO_RESOLUTION)
print("ASSAULT_RUN_TRAINING=", int(RUN_TRAINING))
print("HU009C_POST_TRAINING_READY=", HU009C_POST_TRAINING_READY)
print("training_session_bootstrap_skipped=", TRAINING_SESSION_BOOTSTRAP_SKIPPED)
print("TRAINING_SKIPPED_FINAL_CHECKPOINT_EXISTS=", TRAINING_SKIPPED_FINAL_CHECKPOINT_EXISTS)
print("SESSION_BOOTSTRAP_READY=", session_context is not None)
print("FULL_TRAINING_READY=", FULL_TRAINING_READY)
print("TRAINING_PROFILE=", profile_context.name)
print("project_run_id=", PROJECT_RUN_ID)
print("target_global_step=", TARGET_TIMESTEPS)
print("source_checkpoint_candidate=", SOURCE_CHECKPOINT_PATH)
print("replay_buffer_capacity=", profile_context.replay_buffer_memory.capacity)
print("replay_buffer_estimated_memory_gib=", round(profile_context.replay_buffer_memory.total_gib, 3))
print("device=", selected_device)
print("checkpoint_root=", CHECKPOINT_DIR)
print("tensorboard_root=", TENSORBOARD_DIR)

if RUN_TRAINING:
    print("tracking_mode=", session_context.tracking_mode)
    print("mlflow_run_id=", session_context.mlflow_run_id or "<new run>")
    print("tracking_session_id=", session_context.tracking_session_id)
    print("checkpoint_input=", session_context.checkpoint_input)
    print("restored_global_step=", session_context.restored_expected_step)
    print("bootstrap_commit=", session_context.bootstrap_commit)
    print("config_fingerprint=", session_context.config_fingerprint)
    print("runtime_ram_available_gib=", full_training_gate.ram_available_gib)
    print("runtime_ram_margin_gib=", full_training_gate.ram_margin_gib)
    print("tracking_uri=", session_context.tracking_uri)
    print("full_training_gate_issues=", full_training_gate.issues)
else:
    print("HU009C delivery mode; training session bootstrap skipped.")


## 9. HU009 tracked training session


In [ ]:
if not RUN_TRAINING:
    print("TRAINING_SKIPPED=True; AUTO_RESOLUTION=", AUTO_RESOLUTION)
else:
    import copy

    MLFLOW_TRACKING_PASS = False
    MLFLOW_SESSION_ARTIFACTS = []
    CHECKPOINT_INPUT_LOADED = False
    RESTORED_GLOBAL_STEP = None
    REPLAY_BUFFER_RESTORED = False
    MULTISESSION_CHECKPOINT_RESUME_PASS = False
    EXPERIMENT_MANIFEST_UPDATED = False

    session_config = copy.deepcopy(config)
    session_config.setdefault("training", {})["total_timesteps"] = SESSION_TARGET_TIMESTEPS
    session_config.setdefault("mlflow", {})["tracking_uri"] = session_context.tracking_uri
    session_config.setdefault("mlflow", {})["tracking_mode"] = session_context.tracking_mode
    session_config.setdefault("mlflow", {})["mlflow_run_id"] = session_context.mlflow_run_id
    session_config.setdefault("mlflow", {})["tracking_session_id"] = session_context.tracking_session_id

    mlflow_tracker = MLflowTracker.from_config(session_config)
    mlflow_metadata = mlflow_tracker.start_run(
        project_run_id=session_context.project_run_id,
        tracking_mode=session_context.tracking_mode,
        mlflow_run_id=session_context.mlflow_run_id,
        run_name=session_context.project_run_id,
        tags={"stage": "HU008B"},
        tracking_session_id=session_context.tracking_session_id,
    )
    print("MLflow tracking URI:", mlflow_metadata.tracking_uri)
    print("MLflow experiment name:", mlflow_metadata.experiment_name)
    print("project_run_id:", mlflow_metadata.project_run_id)
    print("mlflow_run_id:", mlflow_metadata.mlflow_run_id)
    print("tracking_session_id:", mlflow_metadata.tracking_session_id)

    try:
        mlflow_tracker.log_run_context(
            config=session_config,
            runtime_info=runtime_info,
            git_commit=session_context.bootstrap_commit,
            git_ref=session_context.bootstrap_ref,
            project_run_id=session_context.project_run_id,
            action_space=str(metadata.action_space),
            observation_dtype=str(obs.dtype),
            runtime=selected_runtime,
            device=selected_device,
        )
        mlflow_tracker.log_config_snapshot(config, artifact_file="config/base_config.json")
        effective_config_artifact = mlflow_tracker.log_session_config(
            session_config,
            tracking_session_id=session_context.tracking_session_id,
        )
        mlflow_tracker.log_runtime_metadata(
            runtime_info=runtime_info,
            git_commit=session_context.bootstrap_commit,
            runtime=selected_runtime,
            tracking_session_id=session_context.tracking_session_id,
        )

        session_summary = run_training_session(
            config=session_config,
            checkpoint_root=session_context.checkpoint_root,
            tensorboard_root=session_context.tensorboard_root,
            run_id=session_context.project_run_id,
            repo_path=PROJECT_ROOT,
            tracking_mode=session_context.tracking_mode,
            checkpoint_input=session_context.checkpoint_input,
            resume_mode=session_context.resume_mode,
            total_timesteps=session_context.target_timesteps,
            device=selected_device,
        )
        session_result = session_summary.as_dict()
        checkpoint_output_reference = session_summary.checkpoint_output_reference
        session_initial_step = int(session_summary.initial_global_step)
        session_final_step = int(session_summary.final_global_step)
        CHECKPOINT_INPUT_LOADED = bool(session_summary.checkpoint_input_loaded)
        RESTORED_GLOBAL_STEP = session_summary.restored_global_step
        REPLAY_BUFFER_RESTORED = bool(session_summary.replay_buffer_restored)
        MULTISESSION_CHECKPOINT_RESUME_PASS = bool(
            session_context.tracking_mode == "resume"
            and CHECKPOINT_INPUT_LOADED
            and RESTORED_GLOBAL_STEP == session_initial_step
            and session_context.restored_expected_step == session_initial_step
            and REPLAY_BUFFER_RESTORED
            and session_final_step > session_initial_step
            and session_summary.checkpoint_input_reference == CHECKPOINT_INPUT_REFERENCE
        )

        eval_env = create_assault_env(session_config, mode="eval", seed=seed + 10_000)
        try:
            evaluation_summary = evaluate_agent(
                eval_env,
                session_summary.agent,
                episodes=EVALUATION_EPISODES,
                epsilon=EVALUATION_EPSILON,
                max_steps_per_episode=EVALUATION_MAX_STEPS,
            )
        finally:
            eval_env.close()

        session_result = session_summary.as_dict()
        session_result["evaluation"] = evaluation_summary.as_dict()
        session_result["session_bootstrap"] = session_context.as_dict()
        session_result["training_profile"] = profile_context.as_dict()
        session_result["full_training_gate"] = full_training_gate.as_dict()

        mlflow_tracker.log_training_summary(session_summary.training, tracking_session_id=session_context.tracking_session_id)
        mlflow_tracker.log_evaluation_summary(evaluation_summary, tracking_session_id=session_context.tracking_session_id)
        mlflow_tracker.log_checkpoint_reference(
            session_summary.checkpoint,
            resume_mode=session_context.resume_mode if session_context.tracking_mode == "resume" else "new",
            project_run_id=session_context.project_run_id,
            checkpoint_input_reference=CHECKPOINT_INPUT_REFERENCE,
            checkpoint_output_reference=checkpoint_output_reference,
            tracking_session_id=session_context.tracking_session_id,
        )
        mlflow_tracker.log_dict_artifact(
            session_result,
            "training_session_summary.json",
            tracking_session_id=session_context.tracking_session_id,
        )
        mlflow_tracker.log_session_metadata(
            tracking_mode=session_context.tracking_mode,
            runtime_info=runtime_info,
            git_commit=session_context.bootstrap_commit,
            git_ref=session_context.bootstrap_ref,
            runtime=selected_runtime,
            device=selected_device,
            initial_global_step=session_initial_step,
            final_global_step=session_final_step,
            session_target_timesteps=session_context.target_timesteps,
            checkpoint_input_reference=CHECKPOINT_INPUT_REFERENCE,
            checkpoint_output_reference=checkpoint_output_reference,
            checkpoint_input_loaded=CHECKPOINT_INPUT_LOADED,
            restored_checkpoint_path=session_summary.restored_checkpoint_path,
            restored_global_step=RESTORED_GLOBAL_STEP,
            replay_buffer_restored=REPLAY_BUFFER_RESTORED,
            resume_mode=session_summary.resume_mode,
            effective_config_artifact=effective_config_artifact,
            tracking_session_id=session_context.tracking_session_id,
            extra={
                "config_fingerprint": session_context.config_fingerprint,
                "training_profile": profile_context.name,
                "replay_buffer_memory": profile_context.replay_buffer_memory.as_dict(),
                "full_training_ready": FULL_TRAINING_READY,
            },
        )
        MLFLOW_SESSION_ARTIFACTS = [
            artifact.path for artifact in mlflow_tracker.list_session_artifacts(session_context.tracking_session_id)
        ]
        queried_run = mlflow_tracker.get_run(mlflow_metadata.mlflow_run_id) if mlflow_metadata.enabled else None
        required_session_artifacts = {
            f"sessions/{session_context.tracking_session_id}/session_metadata.json",
            f"sessions/{session_context.tracking_session_id}/runtime.json",
            f"sessions/{session_context.tracking_session_id}/training_summary.json",
            f"sessions/{session_context.tracking_session_id}/evaluation_summary.json",
            f"sessions/{session_context.tracking_session_id}/effective_config.json",
            f"sessions/{session_context.tracking_session_id}/checkpoint_reference.json",
            f"sessions/{session_context.tracking_session_id}/training_session_summary.json",
        }
        MLFLOW_TRACKING_PASS = bool(
            not mlflow_metadata.enabled
            or (
                queried_run is not None
                and queried_run.info.run_id == mlflow_metadata.mlflow_run_id
                and queried_run.data.params.get("identity.project_run_id") == session_context.project_run_id
                and queried_run.data.tags.get("latest_tracking_session_id") == session_context.tracking_session_id
                and "train/final_global_step" in queried_run.data.metrics
                and "eval/mean_reward" in queried_run.data.metrics
                and required_session_artifacts.issubset(set(MLFLOW_SESSION_ARTIFACTS))
                and (session_context.tracking_mode != "resume" or MULTISESSION_CHECKPOINT_RESUME_PASS)
            )
        )
    except Exception:
        mlflow_tracker.end_run(status="FAILED")
        raise
    else:
        mlflow_tracker.end_run(status="FINISHED")
        experiment_state = update_experiment_state_after_success(
            session_context,
            mlflow_metadata.mlflow_run_id,
            checkpoint_output_reference,
            session_final_step,
        )
        EXPERIMENT_MANIFEST_UPDATED = True

    session_result


## 10. HU009 result gates


In [ ]:
if not RUN_TRAINING:
    print("TRAINING_SKIPPED=True; AUTO_RESOLUTION=", AUTO_RESOLUTION)
else:
    assert session_summary.initial_global_step == session_initial_step
    assert session_summary.final_global_step == session_final_step
    assert session_final_step == session_context.target_timesteps
    assert session_summary.checkpoint.path.exists()
    assert evaluation_summary.episodes == EVALUATION_EPISODES
    assert evaluation_summary.epsilon == EVALUATION_EPSILON
    assert EXPERIMENT_MANIFEST_UPDATED
    if session_context.tracking_mode == "new":
        assert session_initial_step == 0
        assert not CHECKPOINT_INPUT_LOADED
        assert CHECKPOINT_INPUT_REFERENCE is None
        assert session_context.restored_expected_step is None
    elif session_context.tracking_mode == "resume":
        assert CHECKPOINT_INPUT_LOADED
        assert RESTORED_GLOBAL_STEP == session_initial_step
        assert session_context.restored_expected_step == session_initial_step
        assert REPLAY_BUFFER_RESTORED
        assert session_initial_step > 0
        assert session_final_step > session_initial_step
        assert session_summary.checkpoint_input_reference == CHECKPOINT_INPUT_REFERENCE
        assert MULTISESSION_CHECKPOINT_RESUME_PASS
    else:
        raise AssertionError(f"Unsupported tracking mode: {session_context.tracking_mode}")
    if mlflow_config.get("enabled", False):
        assert MLFLOW_TRACKING_PASS, "MLflow tracking validation did not pass."

    bootstrap_diagnostics = inspect_experiment_state(
        BASE,
        session_context.project_run_id,
        config=session_config,
        tracking_uri=session_context.tracking_uri,
    )
    assert bootstrap_diagnostics.ok, bootstrap_diagnostics.as_dict()

    print("HU009 training session status")
    print("TRAINING_PROFILE=", profile_context.name)
    print("FULL_TRAINING_READY=", FULL_TRAINING_READY)
    print("full_training_gate_issues:", full_training_gate.issues)
    print("runtime:", selected_runtime)
    print("device:", selected_device)
    print("tracking_mode:", session_context.tracking_mode)
    print("project_run_id:", mlflow_metadata.project_run_id)
    print("mlflow_run_id:", mlflow_metadata.mlflow_run_id)
    print("tracking_session_id:", mlflow_metadata.tracking_session_id)
    print("checkpoint_input_reference:", CHECKPOINT_INPUT_REFERENCE)
    print("checkpoint_input_loaded:", CHECKPOINT_INPUT_LOADED)
    print("restored_global_step:", RESTORED_GLOBAL_STEP)
    print("restored_expected_step:", session_context.restored_expected_step)
    print("replay_buffer_restored:", REPLAY_BUFFER_RESTORED)
    print("initial_global_step:", session_initial_step)
    print("final_global_step:", session_final_step)
    print("checkpoint_output_reference:", checkpoint_output_reference)
    print("session_target_timesteps:", session_context.target_timesteps)
    print("evaluation_episodes:", evaluation_summary.episodes)
    print("evaluation_mean_reward:", evaluation_summary.mean_reward)
    print("evaluation_epsilon:", evaluation_summary.epsilon)
    print("mlflow_tracking_uri:", mlflow_metadata.tracking_uri)
    print("mlflow_experiment:", mlflow_metadata.experiment_name)
    print("manifest_path:", session_context.manifest_path)
    print("manifest_updated:", EXPERIMENT_MANIFEST_UPDATED)
    print("config_fingerprint:", session_context.config_fingerprint)
    print("observation:", metadata.observation_shape, metadata.observation_dtype)
    print("action_space:", metadata.action_space)
    print("Preflight READY_FOR_TRAINING:", preflight_report.ready_for_training)
    print("training_updates:", session_summary.training.updates_count)
    print("training_initial_step:", session_summary.training.initial_global_step)
    print("training_final_step:", session_summary.training.global_step)
    print("checkpoint_path:", session_summary.checkpoint.path)
    print("checkpoint_size_bytes:", session_summary.checkpoint.size_bytes)
    print("replay_buffer_capacity:", profile_context.replay_buffer_memory.capacity)
    print("replay_buffer_estimated_memory_gib:", round(profile_context.replay_buffer_memory.total_gib, 3))
    print("runtime_ram_available_gib:", full_training_gate.ram_available_gib)
    print("runtime_ram_margin_gib:", full_training_gate.ram_margin_gib)
    print("session_artifacts:", MLFLOW_SESSION_ARTIFACTS)
    print("effective_config_artifact:", effective_config_artifact)
    print("SESSION_BOOTSTRAP_READY=True")
    print("MULTISESSION_CHECKPOINT_RESUME_PASS=", MULTISESSION_CHECKPOINT_RESUME_PASS)
    print("MLFLOW_TRACKING_PASS=", MLFLOW_TRACKING_PASS)
    print("TRAINING_COMPLETE=True")
    print("final_global_step=", session_final_step)
    print("SOURCE_CHECKPOINT_READY=", Path(checkpoint_output_reference).exists())


## 11. HU009C artefactos de entrega

Esta seccion transforma la corrida full existente en artefactos de entrega sin repetir el entrenamiento full. Si el checkpoint, TensorBoard o Google Drive no estan disponibles en el runtime actual, la celda marca la validacion como pendiente en lugar de fabricar resultados.


In [ ]:
from pathlib import Path

COMPACT_MODEL_PATH = Path(os.environ.get("ASSAULT_COMPACT_MODEL_PATH") or (BASE / "models" / PROJECT_RUN_ID / "assault_ddqn_model.pt"))
TENSORBOARD_RUN_DIR = Path(os.environ.get("ASSAULT_TENSORBOARD_RUN_DIR") or (Path(os.environ["ASSAULT_TENSORBOARD_DIR"]) / PROJECT_RUN_ID))
VIDEO_PATH = Path(os.environ.get("ASSAULT_VIDEO_PATH") or (BASE / "videos" / PROJECT_RUN_ID / "assault_ddqn_demo.mp4"))

print("project_run_id:", PROJECT_RUN_ID)
print("source_checkpoint:", SOURCE_CHECKPOINT_PATH)
print("compact_model:", COMPACT_MODEL_PATH)
print("tensorboard_run_dir:", TENSORBOARD_RUN_DIR)
print("video_path:", VIDEO_PATH)
print("SOURCE_CHECKPOINT_READY=", SOURCE_CHECKPOINT_PATH.exists())


## 12. Modelo compacto de inferencia


In [ ]:
compact_model_info = None
compact_agent = None

if SOURCE_CHECKPOINT_PATH.exists():
    compact_model_info = export_inference_model(
        checkpoint_path=SOURCE_CHECKPOINT_PATH,
        output_path=COMPACT_MODEL_PATH,
        project_run_id=PROJECT_RUN_ID,
        config=config,
        source_checkpoint_step=CHECKPOINT_STEP,
        repo_path=PROJECT_ROOT,
        extra_metadata={"config_fingerprint": getattr(session_context, "config_fingerprint", None)},
        overwrite=True,
    )
    compact_agent, loaded_model_info = load_inference_model(
        COMPACT_MODEL_PATH,
        device=selected_device,
        expected_sha256=compact_model_info.sha256,
        expected_project_run_id=PROJECT_RUN_ID,
    )
    print("compact_model_size_bytes:", compact_model_info.size_bytes)
    print("compact_model_sha256:", compact_model_info.sha256)
    print("schema_version:", 1)
    assert compact_model_info.size_bytes < 100 * 1024 * 1024
    print("COMPACT_MODEL_READY=True")
else:
    print("VALIDACION COLAB PENDIENTE: checkpoint full no disponible en este runtime.")


## 13. Figuras TensorBoard reales


In [ ]:
training_figures = []
if TENSORBOARD_RUN_DIR.exists():
    training_figures = prepare_training_figures(TENSORBOARD_RUN_DIR, reward_window=10)
    rendered_figures = plot_training_figures(training_figures)
    for figure in rendered_figures:
        display(figure)
    print("training_figure_count:", len(training_figures))
    print("tensorboard_tags:", sorted({tag for spec in training_figures for tag in spec.tags}))
    print("TENSORBOARD_FIGURES_READY=True")
else:
    print("VALIDACION COLAB PENDIENTE: event files TensorBoard no disponibles en este runtime.")


## 14. Evaluacion y video desde modelo compacto


In [ ]:
compact_evaluation_summary = None
video_summary = None

if COMPACT_MODEL_PATH.exists():
    compact_agent, compact_model_info = load_inference_model(
        COMPACT_MODEL_PATH,
        device=selected_device,
        expected_project_run_id=PROJECT_RUN_ID,
    )
    eval_env = create_assault_env(config, mode="eval", seed=seed + 20_000)
    try:
        compact_evaluation_summary = evaluate_agent(
            eval_env,
            compact_agent,
            episodes=EVALUATION_EPISODES,
            epsilon=EVALUATION_EPSILON,
            max_steps_per_episode=EVALUATION_MAX_STEPS,
        )
    finally:
        eval_env.close()
    print(compact_evaluation_summary.as_dict())
    print("EVALUATION_READY=True")

    video_metadata = {
        "project_run_id": PROJECT_RUN_ID,
        "source_checkpoint_step": CHECKPOINT_STEP,
        "model_sha256": compact_model_info.sha256,
        "epsilon": EVALUATION_EPSILON,
        "training_summary": {"final_global_step": CHECKPOINT_STEP},
    }
    video_max_steps = int(os.environ.get("ASSAULT_VIDEO_MAX_STEPS", "1800"))
    video_summary = generate_assault_demo_video(
        agent=compact_agent,
        env_factory=lambda: create_assault_env(config, mode="eval", seed=seed + 30_000, render_mode="rgb_array"),
        output_path=VIDEO_PATH,
        metadata=video_metadata,
        seed=seed + 30_000,
        epsilon=EVALUATION_EPSILON,
        max_steps=video_max_steps,
        fps=30,
    )
    print(video_summary.as_dict())
    if VIDEO_PATH.exists() and VIDEO_PATH.stat().st_size > 0:
        print("VIDEO_READY=True")
        print("video_path=", VIDEO_PATH)
        print("video_reward=", video_summary.reward)
        print("video_steps=", video_summary.steps)
        print("video_seed=", video_summary.seed)
        print("video_epsilon=", video_summary.epsilon)
        print("video_project_run_id=", video_summary.project_run_id)
        print("video_model_sha256=", video_summary.model_sha256)
        try:
            from IPython.display import Video, display

            display(Video(str(VIDEO_PATH), embed=True))
        except Exception as exc:
            print("VIDEO_INLINE_WARNING:", exc)
            print("Video disponible en:", VIDEO_PATH)
    else:
        print("VIDEO_READY=False")
        print("Video no disponible para visualizacion inline:", VIDEO_PATH)
else:
    print("VALIDACION COLAB PENDIENTE: modelo compacto no disponible para evaluacion/video.")


## 15. Reporte tecnico academico

### Problema y objetivo
Entrenar y entregar un agente para `ALE/Assault-v5`, midiendo recompensa promedio sobre al menos 10 episodios independientes y comparando contra la politica aleatoria documentada en HU001.

### Seleccion del algoritmo
Se usa DDQN porque conserva la familia DQN requerida para observaciones visuales Atari y reduce sobreestimacion al separar seleccion de accion con Online Network y evaluacion con Target Network. HU009C no agrega PER, Dueling, Rainbow ni HPO.

### Entorno y preprocessing
El contrato se obtiene desde `configs/ddqn_config.yaml`: `ALE/Assault-v5`, observacion RGB base, grayscale, resize `84x84`, stack de 4 frames, frameskip efectivo 4, `repeat_action_probability=0.25` y `Discrete(7)`.

### Arquitectura del agente
La politica usa `src.network.QNetwork`: CNN Atari-style con tres capas convolucionales, cabeza fully connected de 512 unidades y salida de 7 Q-values. El entrenamiento original uso Online/Target Network, Replay Buffer uniforme y targets DDQN.

### Hiperparametros efectivos
Los hiperparametros se leen programaticamente desde `config` y el perfil `full`: gamma, learning rate, batch size, Replay Buffer, learning starts, frecuencias de update/Target, epsilon inicial/final y total timesteps.

### Librerias, versiones, hardware y tiempo
Las versiones y hardware se obtienen con `get_runtime_info()`. El tiempo de entrenamiento, pasos, episodios y updates deben provenir de MLflow, JSON de sesion, TensorBoard o checkpoint disponible; no se reemplazan con valores inventados si los artefactos no estan presentes.

### Metricas y maximo tres graficas
Las graficas de entrenamiento son exactamente: recompensa + media movil, loss DDQN, y q_mean + epsilon. Todas se generan desde event files TensorBoard reales mediante `src.reporting`.

### Evaluacion >=10 episodios
La evaluacion de entrega carga un agente nuevo desde el modelo compacto con `load_inference_model(...)` y ejecuta `evaluate_agent(..., epsilon=0.0, episodes=10)` salvo override explicito.

### Comparacion contra baseline
La comparacion usa el baseline aleatorio documentado en HU001/ficha tecnica bajo protocolo comparable. Si el artefacto de baseline no esta disponible en el runtime, la comparacion queda marcada como evidencia pendiente, no hardcodeada como fuente primaria.

### Comportamiento observado
El comportamiento aprendido se analiza a partir de la evaluacion y el MP4 generado desde `render_mode="rgb_array"`; no se afirman estrategias que no puedan observarse.

### Limitaciones
Presupuesto de entrenamiento full, una seed principal, stochasticity de ALE, muestra de evaluacion finita y dependencia de artefactos persistidos en Drive/MLflow/TensorBoard para cerrar AV06-AV10.

### Conclusion
HU009C empaqueta la corrida full en un modelo compacto verificable y evidencia reproducible. La historia queda `[IMPLEMENTADA - VALIDACION COLAB PENDIENTE]` hasta ejecutar exportacion, figuras, evaluacion y video reales con los artefactos de Drive.

### Artefactos de entrega
- Checkpoint full fuente: `SOURCE_CHECKPOINT_PATH`
- Modelo compacto: `COMPACT_MODEL_PATH`
- TensorBoard: `TENSORBOARD_RUN_DIR`
- Video: `VIDEO_PATH`
- Consistencia: checkpoint source -> compact model + checksum -> evaluation >=10 episodes -> TensorBoard figures -> video -> notebook report


In [ ]:
delivery_consistency = {
    "project_run_id": PROJECT_RUN_ID,
    "checkpoint_source": str(SOURCE_CHECKPOINT_PATH),
    "compact_model": compact_model_info.as_dict() if compact_model_info else None,
    "evaluation": compact_evaluation_summary.as_dict() if compact_evaluation_summary else None,
    "tensorboard_figures": [figure.as_dict() for figure in training_figures],
    "video": video_summary.as_dict() if video_summary else None,
    "status": "VALIDACION COLAB PENDIENTE" if not (compact_model_info and compact_evaluation_summary and training_figures and video_summary) else "HU009C_ARTIFACTS_READY",
}
print("HU009C_ARTIFACTS_READY=", delivery_consistency["status"] == "HU009C_ARTIFACTS_READY")
delivery_consistency
